In [0]:
from pyspark.sql.functions import * 
from delta.tables import * 

In [0]:
%run /Workspace/Users/narutosha289@gmail.com/Consolidated_pipeline/Setup/Utilities

In [0]:
dbutils.widgets.text("Catalog","FMCG","Catalog") 
dbutils.widgets.text("Data_Source","customers","Data_Source") 

Catalog = dbutils.widgets.get("Catalog")
Data_Source = dbutils.widgets.get("Data_Source")
BasePath = "s3a://sportsbar-us1/customers/customers.csv"
print(BasePath)

s3a://sportsbar-us1/customers/customers.csv


In [0]:
df = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(BasePath)
    .withColumn("read_timeStamp", current_timestamp()) 
    .select("*","_metadata.file_name","_metadata.file_size")
) 

df.show(10)

+-----------+--------------------+---------+--------------------+-------------+---------+
|customer_id|       customer_name|     city|      read_timeStamp|    file_name|file_size|
+-----------+--------------------+---------+--------------------+-------------+---------+
|     789201|      FitFuel Market|Bengaluru|2026-09-11 15:37:...|customers.csv|     1404|
|     789202|      FitFuel Market|Hyderabad|2026-09-11 15:37:...|customers.csv|     1404|
|     789203|      FitFuel Market|New Delhi|2026-09-11 15:37:...|customers.csv|     1404|
|     789301|Athlete's Choice ...|Bengaluru|2026-09-11 15:37:...|customers.csv|     1404|
|     789303|Athlete's Choice ...|New Delhi|2026-09-11 15:37:...|customers.csv|     1404|
|     789101|     Endurance Foods|Bengalore|2026-09-11 15:37:...|customers.csv|     1404|
|     789102|     Endurance Foods|Hyderabad|2026-09-11 15:37:...|customers.csv|     1404|
|     789103|     Endurance Foods|New Delhi|2026-09-11 15:37:...|customers.csv|     1404|
|     7891

In [0]:
df.write.format("delta").option("enableChangeDataFeed","true").mode("overwrite").saveAsTable(f"{Catalog}.{bronze_schema}.{Data_Source}")

### Silver Processing


In [0]:
# df_bronze = spark.sql("select * from fmcg.bronze.customers") 
df_bronze = spark.sql(f"select * from {Catalog}.{bronze_schema}.{Data_Source}")
df_bronze.show(10)

+-----------+--------------------+---------+--------------------+-------------+---------+
|customer_id|       customer_name|     city|      read_timeStamp|    file_name|file_size|
+-----------+--------------------+---------+--------------------+-------------+---------+
|     789201|      FitFuel Market|Bengaluru|2026-09-11 15:37:...|customers.csv|     1404|
|     789202|      FitFuel Market|Hyderabad|2026-09-11 15:37:...|customers.csv|     1404|
|     789203|      FitFuel Market|New Delhi|2026-09-11 15:37:...|customers.csv|     1404|
|     789301|Athlete's Choice ...|Bengaluru|2026-09-11 15:37:...|customers.csv|     1404|
|     789303|Athlete's Choice ...|New Delhi|2026-09-11 15:37:...|customers.csv|     1404|
|     789101|     Endurance Foods|Bengalore|2026-09-11 15:37:...|customers.csv|     1404|
|     789102|     Endurance Foods|Hyderabad|2026-09-11 15:37:...|customers.csv|     1404|
|     789103|     Endurance Foods|New Delhi|2026-09-11 15:37:...|customers.csv|     1404|
|     7891

In [0]:
# Show Duplicates: 
df_duplicates = df_bronze.groupBy("customer_id").count().filter("count > 1")
df_duplicates.show()

+-----------+-----+
|customer_id|count|
+-----------+-----+
|     789321|    2|
|     789503|    2|
|     789522|    2|
|     789603|    2|
+-----------+-----+



In [0]:
# Count before and after removing the duplicates: 
print("Count before duplicates were dropped: ", df_bronze.count())
df_silver = df_bronze.dropDuplicates(["customer_id"])
print("Count after duplicates were dropped: ", df_silver.count())


Count before duplicates were dropped:  39
Count after duplicates were dropped:  35


In [0]:
# Triming the leading or trailing spaces from the customer's name: 

display(df_silver.filter(df_silver["customer_name"] != trim(df_silver["customer_name"])))

customer_id,customer_name,city,read_timeStamp,file_name,file_size
789121,HydroBoost Nutrition,Hyderabad,2026-09-11T15:37:35.731Z,customers.csv,1404
789401,SprintX nutrition,Bengaluru,2026-09-11T15:37:35.731Z,customers.csv,1404
789420,ZenAthlete foods,null,2026-09-11T15:37:35.731Z,customers.csv,1404
789421,ZenAthlete Foods,Hyderbad,2026-09-11T15:37:35.731Z,customers.csv,1404
789521,PrimeFuel Nutrition,null,2026-09-11T15:37:35.731Z,customers.csv,1404
789702,StaminaX Store,Hyderabad,2026-09-11T15:37:35.731Z,customers.csv,1404


In [0]:
df_silver = df_silver.withColumn("customer_name", trim(df_silver["customer_name"]))

In [0]:
# For checking if the spaces are removed": 

display(df_silver.filter(df_silver["customer_name"] != trim(df_silver["customer_name"])))

customer_id,customer_name,city,read_timeStamp,file_name,file_size


In [0]:
# Data quality fix: Fixing typos: 
df_silver.select('city').distinct().show()

+----------+
|      city|
+----------+
| Bengaluru|
| Hyderabad|
| New Delhi|
| Bengalore|
|Hyderabadd|
|      NULL|
|  Hyderbad|
| NewDelhee|
|  NewDelhi|
|Bengaluruu|
|  NewDheli|
+----------+



In [0]:
city_mapping = {
    'Bengaluruu': 'Bengaluru',
    'Bengalore': 'Bengaluru',

    'Hyderabadd': 'Hyderabad',
    'Hyderbad': 'Hyderabad',

    'NewDelhi': 'New Delhi',
    'NewDheli': 'New Delhi',
    'NewDelhee': 'New Delhi'
}

allowed_cities = ['Bengaluru', 'Hyderabad', 'New Delhi'] 
                  
df_silver = (
    df_silver.replace(city_mapping, subset=['city'])
    .withColumn("city", when(df_silver["city"].isNull(), None)
    .when(df_silver["city"].isin(allowed_cities), df_silver["city"])
    .otherwise(None)
    )
) 

df_silver.select('city').distinct().show()

+---------+
|     city|
+---------+
|Bengaluru|
|Hyderabad|
|New Delhi|
|     NULL|
+---------+



In [0]:
# Fix title casing issue: 

df_silver = df_silver.withColumn(
    "customer_name",
    when(
        df_silver["customer_name"].isNull(),
        None
    ).otherwise(
        initcap(df_silver["customer_name"])
    )
)

df_silver.select('customer_name').distinct().show()

+--------------------+
|       customer_name|
+--------------------+
|      Fitfuel Market|
|Athlete's Choice ...|
|     Endurance Foods|
|Hydroboost Nutrition|
|Macrobite Superfoods|
|      Powersnack Hub|
|   Sprintx Nutrition|
|    Zenathlete Foods|
|Peak Performance ...|
| Primefuel Nutrition|
|       Recovery Lane|
|      Staminax Store|
|Eliteathlete Nutr...|
|      Gameplan Foods|
|   Champion's Choice|
+--------------------+



In [0]:
# Handling missing cities: 

df_silver.filter(df_silver["city"].isNull()).show()

+-----------+-------------------+----+--------------------+-------------+---------+
|customer_id|      customer_name|city|      read_timeStamp|    file_name|file_size|
+-----------+-------------------+----+--------------------+-------------+---------+
|     789403|  Sprintx Nutrition|NULL|2026-09-11 15:37:...|customers.csv|     1404|
|     789420|   Zenathlete Foods|NULL|2026-09-11 15:37:...|customers.csv|     1404|
|     789521|Primefuel Nutrition|NULL|2026-09-11 15:37:...|customers.csv|     1404|
|     789603|      Recovery Lane|NULL|2026-09-11 15:37:...|customers.csv|     1404|
+-----------+-------------------+----+--------------------+-------------+---------+



In [0]:
null_customer_names = ['Sprintx Nutrition', 'Zenathlete Foods', 'Primefuel Nutrition', 'Recovery Lane']
df_silver.filter(df_silver["customer_name"].isin(null_customer_names)).show(truncate=False)

+-----------+-------------------+---------+-------------------------+-------------+---------+
|customer_id|customer_name      |city     |read_timeStamp           |file_name    |file_size|
+-----------+-------------------+---------+-------------------------+-------------+---------+
|789401     |Sprintx Nutrition  |Bengaluru|2026-09-11 15:37:35.73135|customers.csv|1404     |
|789402     |Sprintx Nutrition  |Hyderabad|2026-09-11 15:37:35.73135|customers.csv|1404     |
|789403     |Sprintx Nutrition  |NULL     |2026-09-11 15:37:35.73135|customers.csv|1404     |
|789420     |Zenathlete Foods   |NULL     |2026-09-11 15:37:35.73135|customers.csv|1404     |
|789421     |Zenathlete Foods   |Hyderabad|2026-09-11 15:37:35.73135|customers.csv|1404     |
|789422     |Zenathlete Foods   |New Delhi|2026-09-11 15:37:35.73135|customers.csv|1404     |
|789520     |Primefuel Nutrition|Bengaluru|2026-09-11 15:37:35.73135|customers.csv|1404     |
|789521     |Primefuel Nutrition|NULL     |2026-09-11 15:37:

In [0]:
# Business Confirmation Note: City corrections confirmed by business team
customer_city_fix = {
    # Sprintx Nutrition
    789403: "New Delhi",

    # Zenathlete Foods
    789420: "Bengaluru",

    # Primefuel Nutrition
    789521: "Hyderabad",

    # Recovery Lane
    789603: "Hyderabad"
}

df_fix = spark.createDataFrame(
    [(k, v) for k, v in customer_city_fix.items()],
    ["customer_id", "fixed_city"]
)

display(df_fix)

customer_id,fixed_city
789403,New Delhi
789420,Bengaluru
789521,Hyderabad
789603,Hyderabad


In [0]:
df_silver = (
    df_silver
    .join(df_fix, "customer_id", "left")
    .withColumn(
        "city",
        coalesce("city", "fixed_city")
    )
    .drop("fixed_city")
)

In [0]:
df_silver.filter(df_silver["customer_name"].isin(null_customer_names)).show(truncate=False)

+-----------+-------------------+---------+-------------------------+-------------+---------+
|customer_id|customer_name      |city     |read_timeStamp           |file_name    |file_size|
+-----------+-------------------+---------+-------------------------+-------------+---------+
|789601     |Recovery Lane      |Bengaluru|2026-09-11 15:37:35.73135|customers.csv|1404     |
|789420     |Zenathlete Foods   |Bengaluru|2026-09-11 15:37:35.73135|customers.csv|1404     |
|789421     |Zenathlete Foods   |Hyderabad|2026-09-11 15:37:35.73135|customers.csv|1404     |
|789520     |Primefuel Nutrition|Bengaluru|2026-09-11 15:37:35.73135|customers.csv|1404     |
|789403     |Sprintx Nutrition  |New Delhi|2026-09-11 15:37:35.73135|customers.csv|1404     |
|789521     |Primefuel Nutrition|Hyderabad|2026-09-11 15:37:35.73135|customers.csv|1404     |
|789401     |Sprintx Nutrition  |Bengaluru|2026-09-11 15:37:35.73135|customers.csv|1404     |
|789402     |Sprintx Nutrition  |Hyderabad|2026-09-11 15:37:

In [0]:
# Converting customer_id to string:

df_silver = df_silver.withColumn("customer_id", df_silver["customer_id"].cast("string"))
print(df_silver.printSchema())

root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- read_timeStamp: timestamp (nullable = true)
 |-- file_name: string (nullable = true)
 |-- file_size: long (nullable = true)

None


In [0]:
# Standardizing Customer Attributes to Match Parent Company Data Model

df_silver = (
    df_silver
    # Build final customer column: "CustomerName-City" or "CustomerName-Unknown"
    .withColumn(
        "customer",
        concat_ws("-", "customer_name", coalesce(df_silver["city"], lit("Unknown")))
    )
    
    # Static attributes aligned with parent data model
    .withColumn("market", lit("India"))
    .withColumn("platform", lit("Sports Bar"))
    .withColumn("channel", lit("Acquisition"))
)

In [0]:
df_silver.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .option("mergeSchema", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{Catalog}.{silver_schema}.{Data_Source}")

### GOLD

In [0]:
df_silver = spark.sql(f"select * from {Catalog}.{silver_schema}.{Data_Source}")

# take req cols only
# "customer_id, customer_name, city, read_timestamp, file_name, file_size, customer, market, platform, channel"
df_gold = df_silver.select("customer_id", "customer_name", "city", "customer", "market", "platform", "channel")

In [0]:
df_gold.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{Catalog}.{gold_schema}.sb_dim_{Data_Source}")

### Merging data source with parent


In [0]:
delta_table = DeltaTable.forName(spark, "fmcg.gold.dim_customers")
df_child_customers = spark.table("fmcg.gold.sb_dim_customers").select(
    col("customer_id").alias("customer_code"),
    "customer",
    "market",
    "platform",
    "channel"
)

In [0]:
delta_table.alias("target").merge(
    source=df_child_customers.alias("source"),
    condition="target.customer_code = source.customer_code"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]